# EV Charging Demand Prediction & Usage Pattern Analysis
## Part 2: Regression, Classification, Time Series Forecasting & XAI

### Contents:
1. **Regression:** Predict continuous Demand (kWh) using Linear Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM (evaluated with MAE, MSE, RMSE, R²).
2. **Classification:** Predict High-Demand vs Normal-Demand sessions, evaluate class imbalance techniques (Random Oversampling, SMOTE, ADASYN), and benchmark 7 algorithms.
3. **Time Series Forecasting:** Check ADF stationarity, Moving Average, and Exponential Smoothing (evaluated with MAE & RMSE).
4. **Explainable AI:** Feature Importance, SHAP, and LIME interpretability.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("Libraries ready!")

### 1. Load Processed Dataset & Models

In [ ]:
data_path = os.path.join('..', 'data', 'processed', 'cleaned_charging_records.csv')
df = pd.read_csv(data_path)
print(f"Loaded clean data: {df.shape}")

# Load saved metrics summary
with open(os.path.join('..', 'models', 'evaluation_metrics.json'), 'r') as f:
    metrics = json.load(f)

print(f"Best Regression Model: {metrics['best_regression_model']}")
print(f"Best Classification Model: {metrics['best_classification_model']}")

### 2. Regression Models Performance Comparison

In [ ]:
reg_df = pd.DataFrame(metrics['regression_metrics']).T.sort_values(by='R2', ascending=False)
print("=== Regression Model Comparison (Target: Demand kWh) ===")
reg_df

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=reg_df.index, y=reg_df['R2'], palette='crest')
plt.title('Regression Models R² Score Comparison', fontsize=14, fontweight='bold')
plt.ylabel('R² Score')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 3. Classification Models Benchmark & Imbalance Handling

In [ ]:
clf_df = pd.DataFrame(metrics['classification_metrics']).T.sort_values(by='F1', ascending=False)
print("=== Classification Model Comparison (Target: High Demand) ===")
clf_df

In [ ]:
imb_df = pd.DataFrame(metrics['imbalance_comparison']).T
print("=== Class Imbalance Technique Comparison ===")
imb_df

### 4. Time Series Forecasting Performance

In [ ]:
ts_models = pd.DataFrame(metrics['timeseries_metrics']['Models']).T
print("=== Time Series Forecasting Benchmark ===")
print(f"ADF Test Stationary: {metrics['timeseries_metrics']['Stationarity_ADF']['Is_Stationary']} (p-value: {metrics['timeseries_metrics']['Stationarity_ADF']['p_value']})")
ts_models

### 5. Explainable AI (SHAP & LIME)

In [ ]:
import shap
import lime
from lime.lime_tabular import LimeTabularExplainer

reg_model = joblib.load(os.path.join('..', 'models', 'regression_model.pkl'))
preprocessor = joblib.load(os.path.join('..', 'models', 'preprocessor.pkl'))
shap_sample = pd.read_csv(os.path.join('..', 'models', 'shap_sample_data.csv'))

feature_cols = metrics['categorical_columns'] + metrics['numerical_columns']
X_sample_proc = preprocessor.transform(shap_sample[feature_cols])

cat_encoder = preprocessor.named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(metrics['categorical_columns'])
all_features = list(cat_features) + metrics['numerical_columns']

print("Top 10 Feature Importances:")
feat_imp = pd.Series(reg_model.feature_importances_, index=all_features).sort_values(ascending=False).head(10)
print(feat_imp)